# Maintenance Rehearsal (A) — Data Prep from **evidence positions**

Domain-matched counterpart to `04` (kept, not replaced).

| | `04` (A_generic) | this notebook (A_matched) |
|---|---|---|
| text | `cnn_dailymail`, truncated | project's own train corpora |
| labels | ROUGE oracle vs highlights | positive iff contains evidence for a real question |
| domain | news (42% CNN) | genres actually evaluated |

Same 4 genres as `06b`: news, wiki, narrativeqa, caselaw.

> [!warning] CaseHOLD's answer (1 of 5 holdings) never appears in the text — native `evidence_char_pos` can't label sentences. Fixed via `generate_caselaw_extractive_probes.py`'s synthetic verbatim-verified questions.

> [!warning] Positive rates vary a lot by genre (wiki 3.6x news, `corpus_00`) — §3 searches `keep_fraction` per genre, not just for wiki.


In [ ]:
import json
import sys
from bisect import bisect_left
from pathlib import Path

root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

from src.notebook_setup import setup_project

setup_project()

import datasets
import numpy as np
import pandas as pd
import yaml
from nltk.tokenize import sent_tokenize
from transformers import AutoTokenizer

from src.pipeline.extractive import encode_sentences

## 1. Corpora and the split

20 corpora/genre, disjoint from eval. Split at corpus level, not chunk level.


In [ ]:
GENRES = ["news", "narrativeqa", "caselaw", "wiki"]  # matches 06b's genre list
N_TRAIN_CORPORA, N_VAL_CORPORA, N_TEST_CORPORA = 14, 3, 3
SEED = 20260801

corpora = {}
for genre in GENRES:
    dirs = sorted((root / "data" / "processed" / f"{genre}_train").glob("corpus_*"))
    if not dirs:
        raise SystemExit(f"no corpora for {genre} — run data/scripts/build_{genre}_train_corpora.py")
    corpora[genre] = dirs
    print(f"{genre}: {len(dirs)} corpora")

splits = {"train": [], "val": [], "test": []}
for genre, dirs in corpora.items():
    splits["train"] += dirs[:N_TRAIN_CORPORA]
    splits["val"] += dirs[N_TRAIN_CORPORA:N_TRAIN_CORPORA + N_VAL_CORPORA]
    splits["test"] += dirs[N_TRAIN_CORPORA + N_VAL_CORPORA:
                           N_TRAIN_CORPORA + N_VAL_CORPORA + N_TEST_CORPORA]

for name, dirs in splits.items():
    print(f"{name:5}: {len(dirs)} corpora")

## 2. Chunk, then label from evidence positions

Fixed word-window chunks (not `paginate_semantic`), deliberately.


In [ ]:
CHUNK_MAX_WORDS = yaml.safe_load(open("configs/chunking.yaml", encoding="utf-8"))["max_words"]
MIN_SENTENCES = 2  # nothing to select between in a 1-sentence chunk
print(f"CHUNK_MAX_WORDS = {CHUNK_MAX_WORDS} (from configs/chunking.yaml)")


def sentence_spans(text: str) -> list[tuple[str, int, int]]:
    """(sentence, start_char, end_char), searching forward so repeated
    sentences still map to their own occurrence."""
    spans, cursor = [], 0
    for sentence in sent_tokenize(text):
        start = text.find(sentence, cursor)
        if start < 0:
            continue
        spans.append((sentence, start, start + len(sentence)))
        cursor = start + len(sentence)
    return spans


def chunk_spans(text: str, max_words: int) -> list[tuple[int, int]]:
    """Char spans of consecutive `max_words`-word windows."""
    starts, cursor = [], 0
    for word in text.split():
        i = text.find(word, cursor)
        starts.append(i)
        cursor = i + len(word)
    return [
        (starts[k], starts[k + max_words] if k + max_words < len(starts) else len(text))
        for k in range(0, len(starts), max_words)
    ]


_parsed: dict[Path, list[dict]] = {}


def parse_corpus(corpus_dir: Path) -> list[dict]:
    """Chunks with their sentences and absolute char spans — cached.

    Split out from labelling because §3's bisection re-labels the same
    structure eight times; without the cache that is eight full re-parses of
    every corpus (~2 minutes instead of ~2 seconds).
    """
    if corpus_dir in _parsed:
        return _parsed[corpus_dir]
    text = (corpus_dir / "corpus.txt").read_text(encoding="utf-8")
    chunks = []
    for chunk_index, (start, end) in enumerate(chunk_spans(text, CHUNK_MAX_WORDS)):
        spans = sentence_spans(text[start:end])
        if len(spans) < MIN_SENTENCES:
            continue
        chunks.append(
            {
                "id": f"{corpus_dir.parent.name}/{corpus_dir.name}/chunk{chunk_index:03d}",
                "genre": corpus_dir.parent.name.replace("_train", ""),
                "sentences": [s for s, _, _ in spans],
                "spans": [(start + a, start + b) for _, a, b in spans],
            }
        )
    _parsed[corpus_dir] = chunks
    return chunks


def build_examples(corpus_dir: Path, evidence: list[int]) -> list[dict]:
    """One example per chunk; a sentence is positive iff some evidence
    position falls inside it. `evidence` must be sorted (bisect)."""
    examples = []
    for chunk in parse_corpus(corpus_dir):
        labels = []
        for sentence_start, sentence_end in chunk["spans"]:
            i = bisect_left(evidence, sentence_start)
            labels.append(int(i < len(evidence) and evidence[i] < sentence_end))
        examples.append(
            {
                "id": chunk["id"],
                "genre": chunk["genre"],
                "sentences": chunk["sentences"],
                "labels": labels,
            }
        )
    return examples


def load_evidence(corpus_dir: Path, keep_fraction: float = 1.0, seed: int = SEED) -> list[int]:
    # questions_extractive.csv, if present, is this project's own synthetic
    # verbatim-answer QA (data/scripts/generate_caselaw_extractive_probes.py)
    # — needed for caselaw, whose native questions.csv's evidence_char_pos
    # marks where an *excerpt* begins, not where an answer actually sits in
    # the text (the answer is one of 5 holding options never present in the
    # passage). Using the native position here would silently mislabel
    # sentences as positive based on excerpt boundaries, not evidence — wrong,
    # not just missing. news/narrativeqa never have this file and fall
    # through to questions.csv exactly as before.
    extractive_path = corpus_dir / "questions_extractive.csv"
    questions_path = extractive_path if extractive_path.exists() else corpus_dir / "questions.csv"
    positions = pd.read_csv(questions_path, encoding="utf-8-sig")["evidence_char_pos"]
    positions = positions.astype(int).tolist()
    if keep_fraction < 1.0:
        rng = np.random.default_rng(seed + hash(corpus_dir.name) % 10_000)
        n = max(1, int(round(len(positions) * keep_fraction)))
        positions = list(rng.choice(positions, size=n, replace=False))
    return sorted(positions)


def positive_rate(examples: list[dict]) -> float:
    total = sum(len(e["labels"]) for e in examples)
    return sum(sum(e["labels"]) for e in examples) / total if total else 0.0

## 3. Balance the positive rate across genres

Non-reference genres subsampled per-genre (by bisection) to match news's positive rate.


In [ ]:
reference_genre = "news"
adjust_genres = [g for g in GENRES if g != reference_genre]

# Train corpora only. keep_fraction is a hyperparameter, so fitting it on the
# val/test corpora — even through something as thin as a positive-rate
# statistic — would be choosing a setting with data the model is later scored
# against.
train_dirs = {
    genre: [d for d in splits["train"] if d.parent.name == f"{genre}_train"] for genre in GENRES
}
print({g: len(d) for g, d in train_dirs.items()})

reference_examples = []
for corpus_dir in train_dirs[reference_genre]:
    reference_examples += build_examples(corpus_dir, load_evidence(corpus_dir))
target_rate = positive_rate(reference_examples)
print(f"{reference_genre} positive rate (target): {target_rate:.3f}")


def genre_rate(genre: str, keep_fraction: float) -> tuple[float, list[dict]]:
    examples = []
    for corpus_dir in train_dirs[genre]:
        examples += build_examples(corpus_dir, load_evidence(corpus_dir, keep_fraction))
    return positive_rate(examples), examples


def find_keep_fraction(genre: str) -> float:
    """Bisects keep_fraction until genre's positive rate matches the
    reference's — rate rises monotonically with keep_fraction, so bisection
    (not a linear search)."""
    low, high = 0.01, 1.0
    best_fraction = high
    for _ in range(8):
        mid = (low + high) / 2
        rate, _ = genre_rate(genre, mid)
        print(f"  {genre} keep_fraction {mid:.4f} -> positive rate {rate:.3f}")
        if rate > target_rate:
            high = mid
        else:
            low = mid
        best_fraction = mid
    return best_fraction


KEEP_FRACTIONS: dict[str, float] = {}
for genre in adjust_genres:
    native_rate, _ = genre_rate(genre, 1.0)
    print(f"\n{genre} native positive rate: {native_rate:.3f} (target {target_rate:.3f})")
    if native_rate <= target_rate:
        # Subsampling only ever pushes the rate down further — a genre
        # already at or below the reference has no keep_fraction that fixes
        # an imbalance that isn't there in this direction.
        KEEP_FRACTIONS[genre] = 1.0
        print("  already <= target — keeping all questions (keep_fraction 1.0)")
        continue
    fraction = find_keep_fraction(genre)
    rate, _ = genre_rate(genre, fraction)
    KEEP_FRACTIONS[genre] = fraction
    print(f"{genre} keep_fraction = {fraction:.4f} -> positive rate {rate:.3f} (target {target_rate:.3f})")

print(f"\nKEEP_FRACTIONS = {KEEP_FRACTIONS}")

## 4. Build the three splits

Rebuilt per split so §3's keep-fraction applies everywhere; splits never share a corpus.


In [ ]:
def build_split(dirs: list[Path]) -> list[dict]:
    examples = []
    for corpus_dir in dirs:
        genre = corpus_dir.parent.name.replace("_train", "")
        fraction = KEEP_FRACTIONS.get(genre, 1.0)
        examples += build_examples(corpus_dir, load_evidence(corpus_dir, fraction))
    return examples


split_examples = {name: build_split(dirs) for name, dirs in splits.items()}

for name, examples in split_examples.items():
    n_sentences = [len(e["sentences"]) for e in examples]
    n_positive = [sum(e["labels"]) for e in examples]
    empty = sum(1 for p in n_positive if p == 0)
    print(f"{name:5}: {len(examples):5} chunks | "
          f"sentences/chunk {np.mean(n_sentences):.1f} | "
          f"positive rate {sum(n_positive) / sum(n_sentences):.3f} | "
          f"all-negative chunks {empty} ({empty / len(examples):.0%})")

print("\nper genre (train):")
for genre in GENRES:
    subset = [e for e in split_examples["train"] if e["genre"] == genre]
    print(f"  {genre:5}: {len(subset):5} chunks | positive rate {positive_rate(subset):.3f}")

## 5. Tokenize

Same encoding as `04` §4, so `05` needs only a `DATA_DIR` change.


In [ ]:
MODEL_NAME = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def encode_example(example: dict) -> dict:
    encoded = encode_sentences(example["sentences"], tokenizer)
    n = encoded["n_kept"]
    return {
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "cls_positions": encoded["cls_positions"],
        "labels": example["labels"][:n],
        "n_dropped": len(example["sentences"]) - n,
    }


def encode_all(examples: list[dict]) -> datasets.Dataset:
    rows = [encode_example(e) for e in examples]
    dropped = sum(r["n_dropped"] for r in rows)
    total = sum(len(r["labels"]) + r["n_dropped"] for r in rows)
    print(f"  sentences dropped for length: {dropped} / {total} ({dropped / total:.2%})")
    lost = sum(sum(e["labels"]) - sum(r["labels"]) for e, r in zip(examples, rows))
    print(f"  evidence-positive sentences lost: {lost}")
    return datasets.Dataset.from_list(
        [{k: v for k, v in r.items() if k != "n_dropped"} for r in rows]
    )


encoded = {}
for name, examples in split_examples.items():
    print(name)
    encoded[name] = encode_all(examples)

lengths = sorted(len(r) for r in encoded["train"]["input_ids"])
pct = lambda q: lengths[int(q * (len(lengths) - 1))]  # noqa: E731
print(f"\ntoken length  50th={pct(0.50)}  90th={pct(0.90)}  99th={pct(0.99)}  max={lengths[-1]} / 512")

train
  sentences dropped for length: 11 / 72533 (0.02%)
  evidence-positive sentences lost: 0
val
  sentences dropped for length: 0 / 15485 (0.00%)
  evidence-positive sentences lost: 0
test
  sentences dropped for length: 0 / 15035 (0.00%)
  evidence-positive sentences lost: 0

token length  50th=277  90th=305  99th=357  max=493 / 512


## 6. Save

Separate directory — `04`'s output stays untouched for comparison.


In [ ]:
OUT_DIR = Path("data/processed/rehearsal_maintenance_evidence")
OUT_DIR.mkdir(parents=True, exist_ok=True)

for name, dataset in encoded.items():
    dataset.save_to_disk(str(OUT_DIR / name))

for name, examples in split_examples.items():
    pd.DataFrame(
        [
            {
                "id": e["id"],
                "genre": e["genre"],
                "sentences": json.dumps(e["sentences"]),
                "labels": json.dumps(e["labels"]),
            }
            for e in examples
        ]
    ).to_csv(OUT_DIR / f"{name}_sentences_raw.csv", index=False, encoding="utf-8-sig")

(OUT_DIR / "manifest.json").write_text(
    json.dumps(
        {
            "source": "data/processed/{news,narrativeqa,caselaw,wiki}_train — disjoint from the eval corpora",
            "label": "sentence contains the evidence position of a real question",
            "genres": GENRES,
            "caselaw_note": "uses questions_extractive.csv (synthetic verbatim QA), "
                             "not CaseHOLD's native multiple-choice questions",
            "chunking": f"fixed {CHUNK_MAX_WORDS}-word windows (not paginate_semantic)",
            "reference_genre": reference_genre,
            "keep_fractions": KEEP_FRACTIONS,
            "split": "corpus-level",
            "counts": {k: len(v) for k, v in encoded.items()},
        },
        indent=2,
    ),
    encoding="utf-8",
)

print(f"Saved to: {OUT_DIR}")
for name, dataset in encoded.items():
    print(f"  {name}: {len(dataset)} examples")

## Summary

- Source: 20-corpus train sets (news/narrativeqa/caselaw/wiki), disjoint from eval. Caselaw uses synthetic verbatim QA.
- Label: positive iff sentence contains a real question's evidence.
- Split at corpus level.
- Non-reference genres' questions subsampled to match news's positive rate (per-genre bisection).

### Use

```python
DATA_DIR = Path("data/processed/rehearsal_maintenance_evidence")
OUTPUT_DIR = Path("experiments/rehearsal_maintenance_evidence")
```

### Comparison

| | trained on | expected strong on |
|---|---|---|
| A_generic | `cnn_dailymail`, ROUGE oracle | news |
| A_matched | eval genres, evidence labels | news+narrativeqa+caselaw+wiki |

Compare `recall@k` per genre, and vs B's retention curve.
